# Worked Example: Astigmatic Beam Shaping with Spherical and Cylindrical Lenses

**Problem**: Starting from a circular Gaussian beam (λ=532 nm, w₀=1.4 mm),  
find a 3-lens system (Spherical + CylX + CylY) that achieves  
**w_x ≈ 140 µm, w_y ≈ 700 µm at z = 500 mm** (both as beam waists).

**Lens types**:
| Type | x-axis | y-axis |
|------|--------|--------|
| `spherical` | ThinLens(f) | ThinLens(f) |
| `cyl_x` | ThinLens(f) | identity\* |
| `cyl_y` | identity\* | ThinLens(f) |

\* In the thin-lens limit, the non-focused axis is unaffected.

**Tolerance**: 5% on each axis waist width.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from gbeampro import GaussBeam, OpticalSystem, Propagation, ThinLens
from gbeampro.optimize import find_lens_system_astigmatic, build_xy_systems
import gbeampro.plot as gplot
import gbeampro.analysis as ga
import gbeampro
print('gbeampro version:', gbeampro.__version__)

## Problem Setup

In [ ]:
WL        = 0.532   # µm
W0        = 1.4     # mm  (circular input)
Z_TARGET  = 500.0   # mm
WX_TARGET = 0.140   # mm = 140 µm
WY_TARGET = 0.700   # mm = 700 µm
TOL       = 0.05    # 5% tolerance

beam = GaussBeam.from_waist(wl_um=WL, w0_mm=W0)
print(beam)

# Without any optics:
traj0 = OpticalSystem().add(Propagation(Z_TARGET)).trace(beam, dz=1.0)
print(f'\nWithout optics at z={Z_TARGET} mm:  w = {traj0[-1].w_mm*1e3:.1f} µm')
print(f'Targets:  w_x = {WX_TARGET*1e3:.0f} µm,  w_y = {WY_TARGET*1e3:.0f} µm')

## Optimization

Combination: **Spherical + CylX + CylY**

- `optimize_waist=True`: adds `(z_target/R)²` penalty to ensure actual beam waists at z_target
- `f_bounds=None`: no restriction on focal length (searches 1–10000 mm)
- `min_lens_sep_mm=20.0`: lenses must be at least 20 mm apart

In [ ]:
lens_types = ['spherical', 'cyl_x', 'cyl_y']

specs, residual = find_lens_system_astigmatic(
    beam, Z_TARGET, WX_TARGET, WY_TARGET,
    lens_types=lens_types,
    f_bounds=None,
    optimize_waist=True,
    waist_weight=1.0,
    min_lens_sep_mm=20.0,
    maxiter=1000, popsize=15, tol=1e-12, seed=42,
)

sx, sy = build_xy_systems(beam, specs, Z_TARGET)
wx = sx.trace(beam, dz=1.0)[-1].w_mm * 1e3
wy = sy.trace(beam, dz=1.0)[-1].w_mm * 1e3
print(f'wx={wx:.1f} µm  wy={wy:.1f} µm  rel.err²={residual:.2e}')

## Optimized System

In [ ]:
print(f"{'#':>2}  {'Type':<12}  {'z (mm)':>10}  {'f (mm)':>10}")
print('   ' + '-' * 38)
for i, s in enumerate(specs, 1):
    print(f"{i:>2}  {s['type']:<12}  {s['z_mm']:>10.3f}  {s['f_mm']:>10.3f}")

print(f"\nAt z = {Z_TARGET} mm:")
print(f"  w_x = {wx:.2f} µm  (target {WX_TARGET*1e3:.0f} µm, tol ±{TOL*100:.0f}%)")
print(f"  w_y = {wy:.2f} µm  (target {WY_TARGET*1e3:.0f} µm, tol ±{TOL*100:.0f}%)")
print(f"  err_x = {abs(wx - WX_TARGET*1e3)/(WX_TARGET*1e3)*100:.2f}%")
print(f"  err_y = {abs(wy - WY_TARGET*1e3)/(WY_TARGET*1e3)*100:.2f}%")

## Caustic Plot

In [ ]:
sx, sy = build_xy_systems(beam, specs, Z_TARGET)
traj_x = sx.trace(beam, dz=1.0)
traj_y = sy.trace(beam, dz=1.0)

fig, ax = plt.subplots(figsize=(12, 4))
gplot.plot_system(sx, traj_x, ax, label=f'x-axis  (target {WX_TARGET*1e3:.0f} µm)')
gplot.plot_system(sy, traj_y, ax, label=f'y-axis  (target {WY_TARGET*1e3:.0f} µm)')
ax.axvline(Z_TARGET, color='k', ls=':', lw=1.2)
ax.scatter([Z_TARGET, Z_TARGET], [WX_TARGET*1e3, WY_TARGET*1e3],
           color='k', zorder=6, s=60, label='targets')
ax.set_title('Sph + CylX + CylY  (optimize_waist=True)')
plt.tight_layout()

## Waist Verification

Propagate past z_target to z_target + 200 mm and use `analysis.find_waists` to confirm that actual beam waists exist near z = 500 mm with the required beam widths (within 5% tolerance).

In [ ]:
Z_EXTRA = Z_TARGET + 200.0  # propagate past target to detect waist

sx_ext, sy_ext = build_xy_systems(beam, specs, Z_EXTRA)
traj_x_ext = sx_ext.trace(beam, dz=1.0)
traj_y_ext = sy_ext.trace(beam, dz=1.0)

waists_x = ga.find_waists(traj_x_ext)
waists_y = ga.find_waists(traj_y_ext)

print('x-axis waists:')
for w in waists_x:
    err = abs(w.w_mm - WX_TARGET) / WX_TARGET * 100
    ok = '✓' if err <= TOL * 100 else '✗'
    print(f'  z={w.z_mm:.1f} mm  w₀={w.w_mm*1e3:.2f} µm  err={err:.2f}%  {ok}')

print('\ny-axis waists:')
for w in waists_y:
    err = abs(w.w_mm - WY_TARGET) / WY_TARGET * 100
    ok = '✓' if err <= TOL * 100 else '✗'
    print(f'  z={w.z_mm:.1f} mm  w₀={w.w_mm*1e3:.2f} µm  err={err:.2f}%  {ok}')

# Final pass/fail
x_ok = any(abs(w.w_mm - WX_TARGET)/WX_TARGET <= TOL for w in waists_x)
y_ok = any(abs(w.w_mm - WY_TARGET)/WY_TARGET <= TOL for w in waists_y)
print(f'\nResult: x={"PASS" if x_ok else "FAIL"}  y={"PASS" if y_ok else "FAIL"}')

## System Summary

In [ ]:
print('=== x-axis system ===')
print(sx_ext.summary(beam))
print()
print('=== y-axis system ===')
print(sy_ext.summary(beam))